In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
path = '/content/drive/MyDrive/DM'
os.chdir(path)

Mounted at /content/drive


In [1]:
import pickle
import gzip
import logging
from collections import defaultdict, Counter
from typing import List, Tuple, Dict, Any, Union
from tqdm.auto import tqdm

# Cấu hình logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

class NgramLanguageModel:
    """
    Mô hình N-gram Language Model thống kê truyền thống.
    Hỗ trợ N-gram tổng quát (Bigram, Trigram...) và nhiều chiến lược Smoothing.
    """

    def __init__(self, n: int = 3, smoothing: str = 'laplace', k: float = 1.0, discount: float = 0.75):
        """
        :param n: Kích thước cửa sổ N-gram (2 cho Bigram, 3 cho Trigram).
        :param smoothing: Phương pháp làm mịn ('none', 'laplace', 'add-k', 'kneser-ney').
        :param k: Tham số k cho Add-k (Laplace mặc định k=1.0).
        :param discount: Tham số d (discount) cho Kneser-Ney (thường từ 0.5 đến 0.75).
        """
        assert n >= 2, "Model cần ít nhất n=2 (Bigram)"
        assert smoothing in ['none', 'laplace', 'add-k', 'kneser-ney'], "Smoothing không hợp lệ!"

        self.n = n
        self.smoothing = smoothing
        self.k = 1.0 if smoothing == 'laplace' else k
        self.discount = discount

        # Bảng tần suất
        self.vocab = set()
        self.vocab_size = 0

        # self.counts[context][target] = count
        self.counts = defaultdict(Counter)

        # self.context_totals[context] = tổng số lần context xuất hiện
        self.context_totals = Counter()

        # --- Dành riêng cho Kneser-Ney ---
        self.continuation_counts = Counter()
        self.total_ngram_types = 0

    # ==========================================
    # PHẦN 1 & 5: COUNT N-GRAM & OPTIMIZATION
    # ==========================================
    def fit(self, corpus: List[List[str]], vocab: set):
        """
        Huấn luyện mô hình bằng cách đếm tần suất N-gram từ corpus.
        """
        logger.info(f"Đang huấn luyện {self.n}-gram model với smoothing '{self.smoothing}'...")
        self.vocab = vocab
        self.vocab_size = len(self.vocab)

        for sentence in tqdm(corpus, desc="Counting N-grams"):
            if len(sentence) < self.n:
                continue

            # Trượt cửa sổ N-gram qua từng câu
            for i in range(len(sentence) - self.n + 1):
                window = sentence[i : i + self.n]
                context = tuple(window[:-1])
                target = window[-1]

                self.counts[context][target] += 1
                self.context_totals[context] += 1

                if self.smoothing == 'kneser-ney':
                    # Đếm số lượng context khác nhau đứng trước target w
                    # Cần thiết để tính P_continuation
                    self.continuation_counts[target] += 1
                    self.total_ngram_types += 1

        # Memory Optimization: Convert defaultdict về dict chuẩn để giảm RAM
        logger.info("Đang tối ưu hóa bộ nhớ (Freezing dicts)...")
        self.counts = {ctx: dict(target_counts) for ctx, target_counts in self.counts.items()}
        self.context_totals = dict(self.context_totals)

        if self.smoothing == 'kneser-ney':
            self.continuation_counts = dict(self.continuation_counts)

        logger.info(f"Huấn luyện xong! Ghi nhận {len(self.counts):,} unique contexts.")

    # ==========================================
    # PHẦN 2 & 3: PROBABILITY & SMOOTHING
    # ==========================================
    def predict_proba(self, context: Tuple[str, ...], target: str) -> float:
        """
        Tính xác suất P(target | context).
        """
        # Fallback an toàn nếu context hoặc target chứa từ không có trong vocab
        safe_target = target if target in self.vocab else '<UNK>'
        safe_context = tuple([w if w in self.vocab else '<UNK>' for w in context])

        # Nếu context nhập vào dài hơn thiết lập của mô hình, cắt bớt lấy phần đuôi
        if len(safe_context) > self.n - 1:
            safe_context = safe_context[-(self.n - 1):]
        # Nếu context nhập vào ngắn hơn, đệm thêm <START>
        elif len(safe_context) < self.n - 1:
            pad_len = (self.n - 1) - len(safe_context)
            safe_context = tuple(['<START>'] * pad_len) + safe_context

        count_c_w = self.counts.get(safe_context, {}).get(safe_target, 0)
        count_c = self.context_totals.get(safe_context, 0)

        # 1. No Smoothing (Maximum Likelihood)
        if self.smoothing == 'none':
            if count_c == 0: return 0.0
            return count_c_w / count_c

        # 2 & 3. Laplace / Add-k Smoothing
        elif self.smoothing in ['laplace', 'add-k']:
            return (count_c_w + self.k) / (count_c + self.k * self.vocab_size)

        # 4. Kneser-Ney Smoothing (Interpolated)
        elif self.smoothing == 'kneser-ney':
            # Xác suất Continuation
            p_cont = self.continuation_counts.get(safe_target, 0) / max(1, self.total_ngram_types)

            if count_c == 0:
                # Nếu context chưa từng xuất hiện, trả về hoàn toàn xác suất continuation
                return p_cont

            # Tính Discounted Probability
            discounted_prob = max(count_c_w - self.discount, 0) / count_c

            # Tính Lambda (Trọng số nội suy)
            unique_continuations = len(self.counts.get(safe_context, {}))
            lambda_weight = (self.discount / count_c) * unique_continuations

            return discounted_prob + lambda_weight * p_cont

    # ==========================================
    # PREDICT NEXT (Ứng dụng)
    # ==========================================
    def predict_next(self, context: Tuple[str, ...], top_k: int = 5) -> List[Tuple[str, float]]:
        """
        Dự đoán K từ tiếp theo có xác suất cao nhất.
        Thực hiện brute-force tính xác suất trên toàn bộ Vocabulary.
        """
        probabilities = []
        for word in self.vocab:
            # Bỏ qua token không mang ý nghĩa sinh văn bản
            if word in ['<START>', '<UNK>']:
                continue

            prob = self.predict_proba(context, word)
            probabilities.append((word, prob))

        # Sort giảm dần theo xác suất
        probabilities.sort(key=lambda x: x[1], reverse=True)
        return probabilities[:top_k]

    # ==========================================
    # PHẦN 6: SERIALIZATION (LƯU TRỮ)
    # ==========================================
    def save(self, filepath: str, compressed: bool = True):
        """Lưu model. Dùng gzip để nén vì count dictionary rất tốn dung lượng."""
        logger.info(f"Đang lưu model tại {filepath} (Compressed: {compressed})...")
        open_func = gzip.open if compressed else open

        with open_func(filepath, 'wb') as f:
            pickle.dump(self.__dict__, f, protocol=pickle.HIGHEST_PROTOCOL)
        logger.info("Đã lưu xong!")

    @classmethod
    def load(cls, filepath: str, compressed: bool = True) -> 'NgramLanguageModel':
        """Load model từ file."""
        logger.info(f"Đang load model từ {filepath}...")
        open_func = gzip.open if compressed else open

        with open_func(filepath, 'rb') as f:
            data = pickle.load(f)

        # Khôi phục instance
        model = cls(n=data['n'], smoothing=data['smoothing'], k=data['k'], discount=data['discount'])
        model.__dict__.update(data)

        logger.info("Load model thành công!")
        return model

/home/haloha/.pyenv/versions/3.12.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 1. Load Data đã preprocessed
with open('/content/drive/MyDrive/DM/data/train/train.pkl', 'rb') as f:
    train_corpus = pickle.load(f)
with open('/content/drive/MyDrive/DM/data/train/vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)

# 2. Khởi tạo và Train (Ví dụ: Trigram với Laplace Smoothing)
model = NgramLanguageModel(n=3, smoothing='laplace')
model.fit(train_corpus, vocab)

# 3. Dự đoán (Context là 2 từ trước đó do n=3)
context = ('tôi', 'đang')
top_words = model.predict_next(context, top_k=5)

print(f"Ngữ cảnh: {context}")
for word, prob in top_words:
    print(f" -> {word}: {prob:.6f}")

# 4. Lưu lại
model.save('/content/drive/MyDrive/DM/data/train/trigram_laplace.pkl.gz')

2026-05-12 17:16:59,211 - Đang huấn luyện 3-gram model với smoothing 'laplace'...
Counting N-grams: 100%|████████████| 2187145/2187145 [01:24<00:00, 25925.54it/s]
2026-05-12 17:18:23,585 - Đang tối ưu hóa bộ nhớ (Freezing dicts)...
2026-05-12 17:18:33,334 - Huấn luyện xong! Ghi nhận 8,178,928 unique contexts.
2026-05-12 17:18:33,724 - Đang lưu model tại ./train/trigram_laplace.pkl.gz (Compressed: True)...


Ngữ cảnh: ('tôi', 'đang')
 -> ở: 0.000055
 -> làm_việc: 0.000033
 -> làm: 0.000030
 -> cố_gắng: 0.000030
 -> đi: 0.000021


2026-05-12 17:20:08,654 - Đã lưu xong!


In [ ]:
# Kneser-Ney với hệ số discount mặc định (0.75)
model_kn = NgramLanguageModel(n=3, smoothing='kneser-ney')
model_kn.fit(train_corpus, vocab)

# Dự đoán
context = ('tôi', 'đang')
top_words = model_kn.predict_next(context, top_k=5)
print(f"Ngữ cảnh: {context}")
for word, prob in top_words:
    print(f" -> {word}: {prob:.6f}")

model_kn.save('/content/drive/MyDrive/DM/data/train/trigram_kn.pkl.gz')

2026-05-12 17:20:08,657 - Đang huấn luyện 3-gram model với smoothing 'kneser-ney'...
Counting N-grams: 100%|████████████| 2187145/2187145 [01:35<00:00, 22871.72it/s]
2026-05-12 17:21:44,285 - Đang tối ưu hóa bộ nhớ (Freezing dicts)...
2026-05-12 17:21:54,341 - Huấn luyện xong! Ghi nhận 8,178,928 unique contexts.
2026-05-12 17:21:54,958 - Đang lưu model tại ./train/trigram_kn.pkl.gz (Compressed: True)...


Ngữ cảnh: ('tôi', 'đang')
 -> ở: 0.081617
 -> làm_việc: 0.045037
 -> làm: 0.041059
 -> cố_gắng: 0.040096
 -> đi: 0.025972


2026-05-12 17:23:28,740 - Đã lưu xong!


In [ ]:
# Add-k Smoothing với hệ số k tùy chỉnh (ví dụ: 0.01 để không làm xác suất bị giảm quá mạnh như Laplace)
model_add_k = NgramLanguageModel(n=3, smoothing='add-k', k=0.01)
model_add_k.fit(train_corpus, vocab)

# Dự đoán
context = ('tôi', 'đang')
top_words = model_add_k.predict_next(context, top_k=5)
print(f"Ngữ cảnh: {context}")
for word, prob in top_words:
    print(f" -> {word}: {prob:.6f}")

# Lưu mô hình
model_add_k.save('/content/drive/MyDrive/DM/data/train/trigram_add_k.pkl.gz')

2026-05-12 17:23:28,743 - Đang huấn luyện 3-gram model với smoothing 'add-k'...
Counting N-grams: 100%|████████████| 2187145/2187145 [01:21<00:00, 26921.03it/s]
2026-05-12 17:24:49,988 - Đang tối ưu hóa bộ nhớ (Freezing dicts)...
2026-05-12 17:25:00,276 - Huấn luyện xong! Ghi nhận 8,178,928 unique contexts.
2026-05-12 17:25:00,663 - Đang lưu model tại ./train/trigram_add_k.pkl.gz (Compressed: True)...


Ngữ cảnh: ('tôi', 'đang')
 -> ở: 0.004874
 -> làm_việc: 0.002868
 -> làm: 0.002582
 -> cố_gắng: 0.002582
 -> đi: 0.001722


2026-05-12 17:26:33,508 - Đã lưu xong!
